# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print("\nPublished on:", getattr(metadata, 'datePublished', 'N/A'))
print("\nAuthor(s):", getattr(metadata, 'author', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields, referencing them by their `@id`
record_sets = dataset.metadata.record_set
if not record_sets:
    # Try the alternate attribute
    record_sets = getattr(dataset.metadata, 'recordSet', [])

def get_name(obj):
    # Tries to get the name of an entity if it exists
    return getattr(obj, 'name', getattr(obj, '@id', str(obj)))

all_record_set_ids = []

for rs in record_sets:
    print(f"RecordSet @id: {getattr(rs, '@id', 'N/A')}")
    all_record_set_ids.append(getattr(rs, '@id', None))
    print(f"  Name: {get_name(rs)}")
    fields = getattr(rs, 'field', [])
    # Some record sets may have a single field (not list)
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        print(f"    Field @id: {getattr(f, '@id', 'N/A')}, name: {get_name(f)} (type: {getattr(f, 'data_type', 'N/A')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, record sets may need to be referenced by their `@id`, e.g.:
# record_set_id = 'cr:RecordSet_some_name_here'
# Let's automatically use all discovered record set @ids from the previous block:

record_sets = dataset.metadata.record_set
if not record_sets:
    record_sets = getattr(dataset.metadata, 'recordSet', [])
if not isinstance(record_sets, list):
    record_sets = [record_sets]

record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
dataframes = {}

# Try to load each record set as a dataframe
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set {record_set_id}")
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Error loading records for record set {record_set_id}: {e}")

# If any record set loaded, print its columns and preview records
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    df = dataframes[first_record_set_id]
    print(f"\nColumns in first loaded record set ({first_record_set_id}):\n", df.columns.tolist())
    df.head()
else:
    print("No record set dataframes were loaded. Please check the schema or data availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Pick the first available dataframe if loaded
if dataframes:
    record_set_id = first_record_set_id
    df = dataframes[record_set_id].copy()

    # Show columns to help select numeric fields
    print("Columns available:", df.columns.tolist())

    # Attempt to pick a likely numeric field
    likely_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int32, np.int64]]
    if not likely_numeric_fields:
        # Try to infer numerics by converting
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        likely_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int32, np.int64]]

    if likely_numeric_fields:
        numeric_field = likely_numeric_fields[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (mean value)")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical field, e.g. the first non-numeric
        non_numeric_fields = [col for col in df.columns if col not in likely_numeric_fields]
        if non_numeric_fields:
            group_field = non_numeric_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
    else:
        print("No numeric fields available for analysis in this record set.")
else:
    print("No data available for EDA. Please extract and load records in the previous section.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic visualization of the first numeric field
if dataframes and likely_numeric_fields:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    # Distribution
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=axs[0])
    axs[0].set_title(f'Distribution of {numeric_field}')

    # Boxplot by group field (if available)
    if non_numeric_fields:
        sns.boxplot(x=group_field, y=numeric_field, data=df, ax=axs[1])
        axs[1].set_title(f'{numeric_field} by {group_field}')
        axs[1].tick_params(labelrotation=45)
    else:
        axs[1].remove()

    plt.tight_layout()
    plt.show()
else:
    print("No fields available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded metadata and records from the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.
* We listed available record sets and fields by their `@id` and examined data for exploratory analysis.
* We demonstrated basic EDA steps such as filtering on a numeric field, normalization, grouping, and simple visualizations.
* Extend this notebook by using additional fields and record sets to perform domain-specific analysis relevant to your research goals.